<a href="https://colab.research.google.com/github/e23281-lgtm/Statistical-Learning-e23281/blob/main/Copy_of_Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


Answers

---



###Task 1:

The two-parameter logistic (2PL) Item Response Theory (IRT) model defines the probability that a user with ability $\theta$ correctly answers item $i$:$$P(Y_i = 1 \mid \Theta = \theta) = p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Define theta grid
theta = np.linspace(-4, 4, 500)


def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))


# Define combinations
curves = [
    {"a": 1.5, "b": -1.5, "name": "a = 1.5, b = -1.5 (Easy)", "dash": "solid"},
    {"a": 1.5, "b": 0.0, "name": "a = 1.5, b = 0.0 (Medium)", "dash": "solid"},
    {"a": 1.5, "b": 1.5, "name": "a = 1.5, b = 1.5 (Hard)", "dash": "solid"},
    {"a": 0.6, "b": 0.0, "name": "a = 0.6, b = 0.0 (Low Discrim)", "dash": "dash"},
]

fig = go.Figure()

for c in curves:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=p_i(theta, c["a"], c["b"]),
            mode="lines",
            name=c["name"],
            line=dict(dash=c["dash"], width=2.5),
        )
    )

fig.update_layout(
    title="2PL Item Characteristic Curves (ICCs)",
    xaxis_title="User Ability Parameter (θ)",
    yaxis_title="Probability of Correct Response P(Y=1 | θ)",
    template="plotly_white",
    hovermode="x unified",
    yaxis=dict(range=[0, 1]),
)

fig.show()

Interpretation of Horizontal Shift

Difficulty ($b_i$) as Inflection Point: The parameter $b_i$ represents the difficulty of item $i$, corresponding to the point on the ability axis where $P(Y_i = 1 \mid \Theta = b_i) = 0.5$.Horizontal Translation: Increasing $b_i$ shifts the curve horizontally to the right. A user needs higher ability $\theta$ to achieve a $50\%$ chance of answering a harder question correctly. Decreasing $b_i$ shifts the curve to the left, making it accessible to lower-ability users.

###Task 2:
Likelihood Contribution of a Single ResponseFor a single observation $y_k \in \{0, 1\}$ at step $k$:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$Substituting $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$:$$L(y_k \mid \theta) = \left( \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right)^{1 - y_k}$$Joint Likelihood Function for Response Vector $y^{(k)}$Assuming responses are conditionally independent given ability $\Theta = \theta$:$$L(y^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

###Task 3:

By Bayes' Theorem, the posterior density after observing $y^{(k)}$ is proportional to the product of the prior at step $k$ (which is the posterior state at step $k-1$) and the single-item likelihood at step $k$:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \cdot L(y_k \mid \theta)$$Starting from the initial standard normal prior $f_\Theta^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right)$, the unnormalized posterior at step $k$ is:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = C_k \cdot f_\Theta^{(0)}(\theta) \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$where $C_k = \left( \int_{-\infty}^{\infty} f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) L(y_k \mid \theta) \, d\theta \right)^{-1}$ is the normalizing constant.Task 4: Dynamic Shifting Mechanics

###Task 4:
 When a user answers a question correctly ($y_k = 1$), the likelihood contribution is $L(y_k = 1 \mid \theta) = p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.Monotonic Weighting: The function $p_k(\theta)$ is strictly increasing in $\theta$. Multiplying the prior density $f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$ by an increasing function reweights the density, placing heavier relative weight on higher values of $\theta$.Impact of High Difficulty ($b_k \gg 0$): If an item is difficult, $p_k(\theta)$ is near zero for $\theta < b_k$ and ramps up sharply around $\theta \approx b_k$. A correct response ($y_k = 1$) acts as a strong upward filter, significantly dampening the probability density for $\theta < b_k$ while preserving or boosting density for $\theta > b_k$.Shift of the MAP Estimate: This multiplying filter pulls the mode (peak) of the posterior density substantially to the right (toward higher ability estimates), reflecting a strong positive update in belief

###Task 5:

The discrimination parameter $a_k$ controls the steepness of the item response curve at $\theta = b_k$:$$\frac{d}{d\theta} p_k(\theta) \Big\vert{}_{\theta = b_k} = \frac{a_k}{4}$$Large $a_k$ (High Discrimination): The likelihood function $L(y_k \mid \theta)$ transitions very sharply from $0$ to $1$ (or $1$ to $0$) near $\theta = b_k$. This steep gradient drastically shrinks the posterior variance around the transition point, causing a sharp, high-certainty update with rapid reduction in posterior uncertainty.Small $a_k$ (Low Discrimination): The likelihood curve is flat and broad across the ability range. Multiplying by a flat function alters the shape of the prior very little, leading to a broad posterior with high variance (minimal gain in measurement certainty).

###Task 6:

Because the denominator in the 2PL IRT posterior lacks a closed-form conjugate solution, we maintain the posterior on a discrete grid of ability values.Grid Approximation AlgorithmDefine Grid Bounds: Set a bounded range $\theta \in [\theta_{\min}, \theta_{\max}]$ (e.g., $[-4, 4]$) discretized into $M$ evenly spaced points $\{\theta_1, \theta_2, \dots, \theta_M\}$ with step size $\Delta \theta = \frac{\theta_{\max} - \theta_{\min}}{M-1}$.Initialize Prior Vector: Compute the initial prior density vector $\boldsymbol{f}^{(0)} = [f_1^{(0)}, \dots, f_M^{(0)}]$ where $f_j^{(0)} = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta_j^2}{2}\right)$. Normalize using trapezoidal quadrature:$$\boldsymbol{f}^{(0)} \leftarrow \frac{\boldsymbol{f}^{(0)}}{\sum_{j=1}^M f_j^{(0)} \Delta \theta}$$Sequential Step Update: Upon observing response $y_k$ for item $k = (a_k, b_k)$:Compute likelihood array $\boldsymbol{L}^{(k)}$ where $L_j^{(k)} = [p_k(\theta_j)]^{y_k} [1 - p_k(\theta_j)]^{1 - y_k}$.Calculate unnormalized posterior vector: $\boldsymbol{u}^{(k)} = \boldsymbol{f}^{(k-1)} \odot \boldsymbol{L}^{(k)}$ (element-wise product).Computational Normalization:$$\boldsymbol{f}^{(k)} = \frac{\boldsymbol{u}^{(k)}}{\sum_{j=1}^M u_j^{(k)} \Delta \theta}$$

###Task 7:
Python Simulation & Plotly Visualization Script

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(42)
theta_true = 0.75
n_items = 20

# Define numerical grid
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
d_theta = theta_grid[1] - theta_grid[0]

# Initial Standard Normal Prior
prior = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)
posterior = prior / np.trapz(prior, theta_grid)

# Storage for running estimates
bayes_means = [np.trapz(theta_grid * posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]

# Generate item parameters
b_params = np.random.normal(0, 1, n_items)
a_params = np.random.uniform(0.5, 2.0, n_items)

# 2. Sequential Bayesian Simulation
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # Calculate true success probability and simulate user response
    p_true = 1.0 / (1.0 + np.exp(-a_k * (theta_true - b_k)))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Grid evaluation of likelihood
    p_grid = 1.0 / (1.0 + np.exp(-a_k * (theta_grid - b_k)))
    likelihood = (p_grid**y_k) * ((1.0 - p_grid) ** (1.0 - y_k))

    # Sequential update and normalization
    unnorm_post = posterior * likelihood
    posterior = unnorm_post / np.trapz(unnorm_post, theta_grid)

    # Compute point estimators
    current_mean = np.trapz(theta_grid * posterior, theta_grid)
    current_map = theta_grid[np.argmax(posterior)]

    bayes_means.append(current_mean)
    map_estimates.append(current_map)

# 3. Visualization with Plotly
steps = np.arange(0, n_items + 1)

fig = go.Figure()

# True ability line
fig.add_trace(
    go.Scatter(
        x=[0, n_items],
        y=[theta_true, theta_true],
        mode="lines",
        name=f"True Ability (θ_true = {theta_true})",
        line=dict(color="red", width=2, dash="dash"),
    )
)

# Posterior Mean line
fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_means,
        mode="lines+markers",
        name="Posterior Mean (θ̂_Bayes)",
        line=dict(color="blue", width=2.5),
        marker=dict(size=6),
    )
)

# MAP Estimate line
fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate (θ̂_MAP)",
        line=dict(color="green", width=2, dash="dot"),
        marker=dict(size=5),
    )
)

fig.update_layout(
    title="Sequential Bayesian Estimation of User Ability (n = 20 Items)",
    xaxis_title="Item Response Step (k)",
    yaxis_title="Ability Estimate (θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.02, y=0.98),
)

fig.show()

/tmp/ipykernel_2255/635372761.py:16: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2255/635372761.py:19: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2255/635372761.py:41: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

/tmp/ipykernel_2255/635372761.py:44: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

Answers

---



###Task 1:

The probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution for $\theta \in [0, 1]$ is:$$f_{\Theta}(\theta) = \frac{1}{\text{B}(\alpha, \beta)} \, \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

Plotly Code for Beta Distributions

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define grid for theta over [0, 1]
theta_grid = np.linspace(0.001, 0.999, 500)

# Parameter configurations
states = [
    {
        "alpha": 1,
        "beta": 1,
        "label": "α = 1, β = 1 (Uninformative / Uniform)",
        "color": "gray",
    },
    {
        "alpha": 2,
        "beta": 8,
        "label": "α = 2, β = 8 (Right-Skewed / Low CTR)",
        "color": "crimson",
    },
    {
        "alpha": 8,
        "beta": 2,
        "label": "α = 8, β = 2 (Left-Skewed / High CTR)",
        "color": "royalblue",
    },
]

fig = go.Figure()

for state in states:
    a, b = state["alpha"], state["beta"]
    pdf_vals = beta.pdf(theta_grid, a, b)
    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=pdf_vals,
            mode="lines",
            name=state["label"],
            line=dict(color=state["color"], width=2.5),
        )
    )

fig.update_layout(
    title="Beta Distribution PDFs across Different Shape Parameters (α, β)",
    xaxis_title="Click-Through Rate Parameter (θ)",
    yaxis_title="Probability Density f(θ)",
    template="plotly_white",
    hovermode="x unified",
    xaxis=dict(range=[0, 1]),
)

fig.show()

Interpretation of $\alpha$ and $\beta$ Balancing

$\alpha = 1, \beta = 1$ (Uninformative/Uniform): The PDF is constant across $[0, 1]$. All conversion rates are considered equally likely a priori.$\alpha = 2, \beta = 8$ (Right-Skewed): When $\beta > \alpha$, weight concentrates near zero (mode at $\theta = \frac{2-1}{2+8-2} = 0.125$). This represents a belief that the ad has a low click-through rate.$\alpha = 8, \beta = 2$ (Left-Skewed): When $\alpha > \beta$, weight concentrates toward one (mode at $\theta = 0.875$). This represents a belief that the ad is highly effective.Center of Mass Shift: The prior mean is $E[\Theta] = \frac{\alpha}{\alpha + \beta}$. Increasing $\alpha$ relative to $\beta$ pulls the center of mass to the right (higher expected CTR), while increasing $\beta$ pulls it to the left (lower expected CTR).

###Task 2:

Likelihood Contribution of a Single ResponseFor a single user interaction $y_k \in \{0, 1\}$ at time step $k$:$$L(y_k \mid \theta) = P(Y_k = y_k \mid \Theta = \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$Joint Likelihood Function for Response Vector $y^{(k)}$Assuming user interactions are conditionally independent given $\Theta = \theta$:$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k L(y_i \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{s_k} (1 - \theta)^{k - s_k}$$where $s_k = \sum_{i=1}^k y_i$ is the total number of observed clicks up to step $k$.

###Task 3:

Derivation of Recursive Posterior DensityBy Bayes' Theorem, the posterior at step $k$ given prior state $\text{Beta}(\alpha_{k-1}, \beta_{k-1})$ and observation $y_k$ is:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)}) \cdot L(y_k \mid \theta)$$Substitute the previous step's prior and the single-step Bernoulli likelihood:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \left[ \frac{1}{\text{B}(\alpha_{k-1}, \beta_{k-1})} \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] \cdot \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right]$$Group powers of $\theta$ and $(1 - \theta)$:$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$Analytical Proof of ConjugacyThe functional form above is unnormalized, but matches the kernel of a Beta distribution: $\theta^{\alpha_k - 1} (1 - \theta)^{\beta_k - 1}$.Therefore, the posterior remains strictly in the Beta family with simple closed-form arithmetic parameter updates:$$\alpha_k = \alpha_{k-1} + y_k$$$$\beta_k = \beta_{k-1} + (1 - y_k)$$Posterior MeanFor any $\text{Beta}(\alpha_k, \beta_k)$ distribution, the expectation is calculated directly in closed form:$$\hat{\theta}_{\text{Bayes}}^{(k)} = E[\Theta \mid Y^{(k)} = y^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

###Task 4:

Observed Click ($y_k = 1$):Updates: $\alpha_k = \alpha_{k-1} + 1$, $\beta_k = \beta_{k-1}$.Shift: Multiplying by $\theta^1 (1-\theta)^0 = \theta$ shifts weight toward higher values of $\theta$, increasing both the Posterior Mean and the MAP estimate.Observed Non-Click ($y_k = 0$):Updates: $\alpha_k = \alpha_{k-1}$, $\beta_k = \beta_{k-1} + 1$.Shift: Multiplying by $\theta^0 (1-\theta)^1 = (1-\theta)$ shifts weight toward lower values of $\theta$, reducing both point estimates.Contrast with Non-Conjugate Setups (e.g., 2PL IRT Model)Conjugate Setups (Beta-Binomial): The posterior has a known, named distribution family. Updating requires simple addition of counts ($\alpha_k, \beta_k$). Computation is $O(1)$ without integration.Non-Conjugate Setups (2PL IRT): The product of a Normal prior and a logistic sigmoid likelihood yields a posterior that does not belong to standard parametric families. Evaluating the normalization integral $C_k = \int p(\theta) L(y \mid \theta) \, d\theta$ requires numerical grid approximation or MCMC sampling.

###Task 5:

For a $\text{Beta}(\alpha_k, \beta_k)$ posterior at step $k$:Running Posterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$):$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$Running Maximum A Posteriori ($\hat{\theta}_{\text{MAP}}^{(k)}$):The mode of a $\text{Beta}(\alpha_k, \beta_k)$ distribution (for $\alpha_k, \beta_k > 1$) is:$$\hat{\theta}_{\text{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$$(Note: For $\alpha_0 = 1, \beta_0 = 1$ at step $0$, the uniform prior is flat, so any point in $[0,1]$ is a mode; conventionally, $\hat{\theta}_{\text{MAP}}^{(0)} = 0.5$.)

###Task 6:

Python Simulation & Plotly Visualization Script

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(42)
theta_true = 0.35
n_impressions = 100

# Base prior parameters: Beta(1, 1) - Uniform
alpha_k = 1
beta_k = 1

# Tracking arrays
bayes_means = [alpha_k / (alpha_k + beta_k)]
map_estimates = [0.5]  # Standard midpoint mode assignment for Beta(1,1)

# 2. Sequential Beta-Binomial Update Simulation
for k in range(1, n_impressions + 1):
    # Simulate single Bernoulli user response
    u = np.random.uniform(0, 1)
    y_k = 1 if u < theta_true else 0

    # Exact closed-form parameter updates
    alpha_k += y_k
    beta_k += 1 - y_k

    # Compute point estimates directly
    mean_k = alpha_k / (alpha_k + beta_k)
    map_k = (
        (alpha_k - 1) / (alpha_k + beta_k - 2)
        if (alpha_k > 1 and beta_k > 1)
        else mean_k
    )

    bayes_means.append(mean_k)
    map_estimates.append(map_k)

# 3. Visualization with Plotly
steps = np.arange(0, n_impressions + 1)

fig = go.Figure()

# True CTR line
fig.add_trace(
    go.Scatter(
        x=[0, n_impressions],
        y=[theta_true, theta_true],
        mode="lines",
        name=f"True CTR (θ_true = {theta_true})",
        line=dict(color="red", width=2, dash="dash"),
    )
)

# Posterior Mean line
fig.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_means,
        mode="lines+markers",
        name="Posterior Mean (θ̂_Bayes)",
        line=dict(color="royalblue", width=2.5),
        marker=dict(size=4),
    )
)

# MAP Estimate line
fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate (θ̂_MAP)",
        line=dict(color="forestgreen", width=2, dash="dot"),
        marker=dict(size=4),
    )
)

fig.update_layout(
    title="Sequential Bayesian Beta-Binomial CTR Estimation (n = 100 Impressions)",
    xaxis_title="Impression Step (k)",
    yaxis_title="Click-Through Rate Estimate (θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(x=0.65, y=0.98),
)

fig.show()